# GeoQuery Public API — Demo

A living walkthrough of every endpoint under `/api/public/v1/`. Re-run this
notebook against a running GeoQuery dev server to see the API respond for
real — as endpoints are added or change shape, update the matching cell
here rather than letting this drift out of sync.

**Status:** the public API is open (no API key required) during its beta
period. Once account-based access launches, the request cells below will
need an `Authorization: Api-Key ...` header — update this notebook at that
point too.

For the full, always-current reference, see the auto-generated docs at
[`/api/public/v1/docs/`](http://localhost:8000/api/public/v1/docs/) (Swagger
UI) or the raw schema at `/api/public/v1/schema/`. This notebook is the
narrated, run-it-yourself companion to that reference.

## 1. Setup

Assumes the GeoQuery dev stack is running locally (`docker compose up -d`)
so the backend is reachable at `localhost:8000`. Point `BASE_URL` elsewhere
to hit a different environment.

In [ ]:
%pip install requests -q

In [ ]:
import json

import requests

BASE_URL = "http://localhost:8000/api/public/v1"


def call(method, path, **kwargs):
    """Make a request and print status + pretty-printed JSON body."""
    response = requests.request(method, f"{BASE_URL}{path}", **kwargs)
    print(f"{method} {path} -> {response.status_code}")
    body = response.json()
    print(json.dumps(body, indent=2)[:2000])
    return body

## 2. Browse datasets

`GET /datasets/` — flat list of every active, public dataset.

In [ ]:
datasets = call("GET", "/datasets/")
print(f"\n{len(datasets)} dataset(s) available")

## 3. Get a single dataset by name

`GET /datasets/{name}/` — full detail for one dataset. Uses the first
dataset from the list above, if any exist in this environment.

In [ ]:
if datasets:
    dataset_name = datasets[0]["name"]
    call("GET", f"/datasets/{dataset_name}/")
else:
    print("No datasets in this environment yet — skipping.")

## 4. Browse dataset categories

`GET /datasets/categories/` — deduplicated tags across all active, public
datasets, useful for building a category filter UI.

In [ ]:
call("GET", "/datasets/categories/")

## 5. Search boundaries (autocomplete)

`GET /boundaries/autocomplete/?q=&limit=` — search active, public boundary
(feature collection) names/titles/descriptions.

In [ ]:
boundaries = call("GET", "/boundaries/autocomplete/", params={"q": "", "limit": 5})
print(f"\n{len(boundaries)} boundary/boundaries returned")

## 6. Get a boundary by name (with feature IDs)

`GET /boundaries/{name}/` — same fields as autocomplete, plus `feature_ids`:
every `Feature` ID belonging to this boundary collection. This is how a
public API consumer gets real IDs to pass into `/datasets/coverage/`
below — autocomplete deliberately never exposes an ID field on its own, to
avoid leaking internal database primary keys.

In [ ]:
if boundaries:
    boundary_name = boundaries[0]["name"]
    boundary = call("GET", f"/boundaries/{boundary_name}/")
    feature_ids = boundary["feature_ids"]
    print(f"\n{len(feature_ids)} feature ID(s) for '{boundary_name}'")
else:
    feature_ids = []
    print("No boundaries in this environment yet — skipping.")

## 7. Check dataset coverage for boundary IDs

`POST /datasets/coverage/` — given a list of boundary (feature) IDs, returns
the datasets with *confirmed* spatial coverage for at least one of them.
Uses the real `feature_ids` resolved from the boundary detail call above —
chaining autocomplete → boundary detail → coverage is now possible entirely
through the public API.

**Note:** an empty/missing `featureIds` list is treated by this endpoint as
"no filter" and returns *every* active, public dataset — not "no coverage".
So if the chosen boundary happens to have zero features, we skip the call
below rather than showing that unfiltered dump as if it were a real
coverage result for this boundary.

In [ ]:
if feature_ids:
    call("POST", "/datasets/coverage/", json={"featureIds": feature_ids})
else:
    print("No feature IDs resolved — skipping (an empty list would return all datasets, unfiltered).")

## 8. Boundary presets

`GET /boundaries/presets/` — curated filter presets (e.g. "geoBoundaries
ADM1") for batch-selecting boundaries client-side, defined in
`backend/config/boundary_presets.yaml`.

In [ ]:
call("GET", "/boundaries/presets/")

## Keeping this notebook current

When an endpoint is added, removed, or changes shape in
`backend/public_api/`, update the matching section above in the same PR —
treat this notebook as part of the public API's contract, not an
afterthought. `/api/public/v1/schema/` is the source of truth if this
notebook and the live API ever disagree.